# V6.0 DF Experiment — Log-Barrier ORCU + KL Fix + 10-Fold CV + 5-Seed

**V6 changes from V5:** ORCU hinge→log-barrier | kl 0.05→0.07 | lambda_reg 0.005→0.10 | Seeds 3→5 | Gradient clipping

**Experiment (110 runs, ~33h on T4 x2):**
| Phase | Content | Runs | Time |
|-------|---------|:----:|------|
| 2 | Single-seed baseline | 5 | ~1.5h |
| 3A | EDL KL sweep | 15 | ~4.5h |
| 3B | EDL+ORCU λ sweep | 9 | ~2.5h |
| 4 | 5-Seed stability | 25 | ~7.5h |
| 5 | **10-Fold CV (PAPER)** | **50** | **~15h** |
| 6 | A6 ablation | 6 | ~2h |

**▶ Use Kaggle "Run All" to execute from top to bottom.**

## 1. Environment Setup

In [ ]:
!pip install transformers scikit-learn openpyxl scipy pyyaml -q
print("Packages installed.")

## 2. Clone Repository

In [ ]:
import os, subprocess, shutil
REPO_DIR = "/kaggle/working/fluorosis_project"
if os.path.isdir(os.path.join(REPO_DIR, "06_Implementation")):
    print("Repository already cloned.")
else:
    print("Cloning repository...")
    if os.path.isdir(REPO_DIR):
        shutil.rmtree(REPO_DIR)
    r = subprocess.run(["git","clone","https://github.com/XiaoHG/fluorosis_project.git",REPO_DIR],
                       capture_output=True, text=True)
    if r.returncode != 0:
        raise RuntimeError(f"Git clone failed: {r.stderr}")
    print("Clone successful.")

## 3. Imports & Constants

In [ ]:
import sys, json, copy, random
import numpy as np
import torch, torch.nn as nn, torch.nn.functional as F, torch.optim as optim
from torch.optim.lr_scheduler import CosineAnnealingLR, LinearLR, SequentialLR

print("Caching ViT-Base weights...")
from transformers import ViTModel
_ = ViTModel.from_pretrained("google/vit-base-patch16-224-in21k")
print("Done.")

CODE_ROOT = os.path.join(REPO_DIR, "06_Implementation")
DATA_ROOT = "/kaggle/input/datasets/hgxiao/fluorosis_df_pro"
OUTPUT_DIR = "/kaggle/working"
sys.path.insert(0, CODE_ROOT)

from data import create_dataloaders, DFDataset, get_transforms
from models import build_model
from losses import CombinedLoss
from eval import compute_metrics

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

# V6 Constants
EPOCHS = 100
SEEDS_V6   = [42, 123, 456, 789, 1024]
SWEEP_SEEDS = [42, 123, 456]
MODES_V6   = ["ce","cumulative","sord","edl","edl_orcu"]
V6_KL         = 0.07
V6_CAP        = 0.5
V6_LAMBDA_REG = 0.10
print(f"V6: kl={V6_KL} cap={V6_CAP} lambda_reg={V6_LAMBDA_REG} log_barrier=True")

## 4. Data Verification

In [ ]:
print("=== Data Verification ===")
for split in ["train","val","test"]:
    ds = DFDataset(DATA_ROOT, split=split)
    labels = np.array([ds[i][1] for i in range(len(ds))])
    dist = {f"G{k}": int((labels == k).sum()) for k in range(4)}
    print(f"  DF {split:5s}: {len(ds):3d} samples | {dist}")
print("OK.")

## 5. Utility Functions

In [ ]:
# set_seed
def set_seed(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed); torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

# EarlyStopping
class EarlyStopping:
    def __init__(self, patience=15, min_delta=0.001, mode="max"):
        self.patience = patience; self.min_delta = min_delta; self.mode = mode
        self.best = -float("inf") if mode == "max" else float("inf")
        self.counter = 0; self.early_stop = False; self.best_epoch = 0
        self.best_state_dict = None
    def __call__(self, metric, epoch, model=None):
        improved = (metric > self.best + self.min_delta) if self.mode == "max" else (metric < self.best - self.min_delta)
        if improved:
            self.best = metric; self.counter = 0; self.best_epoch = epoch
            if model is not None:
                self.best_state_dict = {k: v.cpu().clone() for k, v in model.state_dict().items()}
        else:
            self.counter += 1
            if self.counter >= self.patience: self.early_stop = True
        return self.early_stop
    def restore(self, model):
        if self.best_state_dict is not None: model.load_state_dict(self.best_state_dict)

# evaluate_model
@torch.no_grad()
def evaluate_model(model, dataloader):
    model.eval()
    all_alpha, all_z, all_targets = [], [], []
    for images, targets in dataloader:
        images, targets = images.to(device), targets.to(device)
        alpha, z = model(images)
        if alpha is not None: all_alpha.append(alpha.cpu())
        all_z.append(z.cpu()); all_targets.append(targets.cpu())
    alpha = torch.cat(all_alpha) if all_alpha else None
    return compute_metrics(alpha, torch.cat(all_z, dim=0), torch.cat(all_targets, dim=0))

print("Utility functions ready.")

## 6. Training Function

In [ ]:
def train_model_v6(task, data_root, output_dir, mode="edl_orcu",
                   batch_size=32, epochs=100,
                   lr_backbone=1e-4, lr_head=1e-3, weight_decay=0.05,
                   warmup_epochs=5,
                   kl_lambda=None, kl_anneal_cap=None,
                   lambda_orcu=0.10, lambda_reg=0.10, orcu_t=3.0,
                   stage_1_epochs=3, stage_2_epochs=10,
                   use_log_barrier=True, patience=None, seed=42):
    """V6 training: log-barrier ORCU, EarlyStopping, gradient clipping."""
    if patience is None: patience = 25 if mode == "edl" else 15
    if kl_lambda is None: kl_lambda = V6_KL
    if kl_anneal_cap is None: kl_anneal_cap = V6_CAP
    set_seed(seed)
    os.makedirs(output_dir, exist_ok=True)

    train_loader, val_loader, test_loader = create_dataloaders(data_root, task=task, batch_size=batch_size, num_workers=2)
    print(f"Data: train={len(train_loader.dataset)} val={len(val_loader.dataset)} test={len(test_loader.dataset)}")

    model = build_model(task=task, mode=mode); model.to(device)
    print(f"Model: {sum(p.numel() for p in model.parameters()):,} params | Mode: {mode} | Seed: {seed}")
    if mode in ("edl","edl_orcu"):
        print(f"  V6: kl={kl_lambda} cap={kl_anneal_cap} | log_barrier={use_log_barrier} lambda_reg={lambda_reg}")

    # Criterion
    if mode == "ce":
        def crit(a,z,t,e): loss=F.nll_loss(F.log_softmax(z,dim=-1),t); return loss,{"stage":0,"L_ce":loss.item()}
    elif mode == "cumulative":
        from losses.cumulative_loss import CumulativeLoss
        cf=CumulativeLoss(num_classes=4)
        def crit(a,z,t,e):
            ol=model.ordinal_logits if hasattr(model,"ordinal_logits") else z
            loss=cf(ol,t); return loss,{"stage":0,"L_cum":loss.item()}
    elif mode == "sord":
        from losses.orcu_loss import ORCULoss
        sf=ORCULoss(num_classes=4,t=orcu_t,lambda_reg=0.0)
        def crit(a,z,t,e): loss=sf(z,t); return loss,{"stage":0,"L_sord":loss.item()}
    elif mode == "edl":
        from losses.edl_loss import EDLLoss
        ef=EDLLoss(num_classes=4,kl_lambda=kl_lambda,kl_anneal_cap=kl_anneal_cap)
        def crit(a,z,t,e): loss=ef(a,t,e,epochs); return loss,{"stage":0,"L_edl":loss.item()}
    else:
        crit=CombinedLoss(num_classes=4,lambda_orcu=lambda_orcu,lambda_kl=kl_lambda,
                          kl_anneal_cap=kl_anneal_cap,orcu_t=orcu_t,orcu_lambda_reg=lambda_reg,
                          stage_1_epochs=stage_1_epochs,stage_2_epochs=stage_2_epochs,total_epochs=epochs)

    # Optimizer
    bb,hd=[],[]
    for n,p in model.named_parameters():
        if not p.requires_grad: continue
        (bb if n.startswith("backbone") else hd).append(p)
    opt=optim.AdamW([{"params":bb,"lr":lr_backbone},{"params":hd,"lr":lr_head}],weight_decay=weight_decay)
    ws=LinearLR(opt,start_factor=0.1,total_iters=warmup_epochs)
    cs=CosineAnnealingLR(opt,T_max=epochs-warmup_epochs)
    sch=SequentialLR(opt,schedulers=[ws,cs],milestones=[warmup_epochs])

    # Training Loop
    es=EarlyStopping(patience=patience,mode="max")
    bva,bst,be=0.0,None,0; stopped=False; hist=[]
    for epoch in range(epochs):
        model.train()
        el={"L_ce":0,"L_cum":0,"L_sord":0,"L_edl":0,"L_orcu":0,"L_total":0}
        es_,nb_=0,0
        for im,tg in train_loader:
            im,tg=im.to(device),tg.to(device)
            a,z=model(im); loss,li=crit(a,z,tg,epoch)
            opt.zero_grad(); loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(),max_norm=1.0)
            opt.step()
            es_=li.get("stage",0)
            for k in el:
                if k in li: el[k]+=li[k]
            nb_+=1
        sch.step()
        for k in el: el[k]/=max(nb_,1)
        vm=evaluate_model(model,val_loader)
        if vm["acc"]>bva: bva=vm["acc"]; bst=copy.deepcopy(model.state_dict()); be=epoch
        hist.append({"epoch":epoch,"stage":es_,"train_loss":el,
                     "val_metrics":{k:v for k,v in vm.items() if isinstance(v,(float,int)) and not k.startswith("evidence_class_")}})
        lk=next((k for k in ["L_total","L_ce","L_cum","L_sord","L_edl","L_orcu"] if el.get(k,0)!=0),"L_total")
        if (epoch+1)%10==0 or epoch==0 or epoch==epochs-1:
            print(f"[E{epoch+1:3d}/{epochs}] S={es_} | {lk}={el.get(lk,0):.4f} | "
                  f"Acc={vm["acc"]:.4f} F1={vm["macro_f1"]:.4f} QWK={vm["qwk"]:.4f} ECE={vm["ece"]:.4f} ES={es.counter}/{patience}")
        if es(vm["acc"],epoch,model):
            print(f"  EarlyStop@{epoch+1} (best: ep{be+1} acc={bva:.4f})")
            stopped=True; break
    if not stopped:
        print(f"  Done {epochs} ep (best: ep{be+1} acc={bva:.4f})")

    model.load_state_dict(bst)
    tm=evaluate_model(model,test_loader)
    torch.save({"model_state_dict":bst,"test_metrics":tm,"best_epoch":be,"stopped_early":stopped},os.path.join(output_dir,"best.pt"))
    ser={}
    for k,v in tm.items():
        if isinstance(v,(float,int,bool)): ser[k]=v
        elif isinstance(v,(np.floating,np.integer)): ser[k]=float(v)
        elif isinstance(v,np.ndarray): ser[k]=v.tolist()
        elif isinstance(v,list): ser[k]=v
    with open(os.path.join(output_dir,"test_metrics.json"),"w") as f: json.dump(ser,f,indent=2)
    with open(os.path.join(output_dir,"history.json"),"w") as f: json.dump(hist,f,indent=2)
    from eval import export_predictions, save_predictions
    pd=export_predictions(model,test_loader,device)
    save_predictions(pd,os.path.join(output_dir,f"v6_{mode}_seed{seed}_preds.npz"),mode=mode,task=task,seed=seed)
    print(f"
[Test] Acc={tm["acc"]:.4f} F1={tm["macro_f1"]:.4f} WgtF1={tm["weighted_f1"]:.4f} QWK={tm["qwk"]:.4f} ECE={tm["ece"]:.4f} BestEp={be+1} ES={stopped}")
    if "per_class_f1" in tm:
        p=tm["per_class_f1"]
        print(f"  Per-Class F1: N={p[0]:.4f} Mi={p[1]:.4f} Mo={p[2]:.4f} Se={p[3]:.4f}")
    return tm,hist

print("train_model_v6() ready.")

## 7. Phase 2: Single-Seed Baseline (5 runs, ~1.5h)

In [ ]:
print("="*70); print("DF V6 Phase 2 — Single-Seed Baseline (s42)"); print("="*70)
p2={}
for mode in MODES_V6:
    print(f"
  {mode} | seed=42")
    out=os.path.join(OUTPUT_DIR,f"v6_p2_{mode}_s42")
    m,_=train_model_v6("df",DATA_ROOT,out,mode=mode,epochs=EPOCHS,seed=42)
    p2[mode]={k:v for k,v in m.items() if isinstance(v,(float,int,bool,list))}
with open(os.path.join(OUTPUT_DIR,"v6_phase2_baseline.json"),"w") as f: json.dump(p2,f,indent=2)
print("
"+"="*70); print("PHASE 2 RESULTS"); print("="*70)
for mode in MODES_V6:
    m=p2[mode]
    print(f"{mode:<14} Acc={m.get("acc",0):.4f} F1={m.get("macro_f1",0):.4f} QWK={m.get("qwk",0):.4f} ECE={m.get("ece",0):.4f}")
print("Phase 2 done.")

## 8. Phase 3A: EDL KL Sweep (15 runs, ~4.5h)

In [ ]:
print("="*70); print("DF V6 Phase 3A — EDL KL Sweep"); print("="*70)
KL_VALS=[0.03,0.05,0.07,0.10,0.15]
p3a=[]
for kl in KL_VALS:
    for seed in SWEEP_SEEDS:
        print(f"
  EDL kl={kl} s{seed}")
        out=os.path.join(OUTPUT_DIR,f"v6_p3a_edl_kl{kl}_s{seed}")
        m,_=train_model_v6("df",DATA_ROOT,out,mode="edl",kl_lambda=kl,epochs=EPOCHS,seed=seed)
        r={"kl_lambda":kl,"seed":seed,"acc":m["acc"],"macro_f1":m["macro_f1"],"qwk":m["qwk"],"ece":m["ece"],"pct_unimodal":m["pct_unimodal"]}
        if "per_class_f1" in m: r["per_class_f1"]=m["per_class_f1"]
        p3a.append(r)
with open(os.path.join(OUTPUT_DIR,"v6_phase3a_kl_sweep.json"),"w") as f: json.dump(p3a,f,indent=2)
for kl in KL_VALS:
    s=[r for r in p3a if r["kl_lambda"]==kl]
    print(f"kl={kl:.2f}: Acc={np.mean([x["acc"] for x in s]):.4f}+/-{np.std([x["acc"] for x in s]):.4f}")
best=max(p3a,key=lambda x:x["acc"])
print(f"
Best: kl={best["kl_lambda"]} s{best["seed"]} Acc={best["acc"]:.4f}")
print("Phase 3A done.")

## 9. Phase 3B: EDL+ORCU Lambda Sweep (9 runs, ~2.5h)

In [ ]:
print("="*70); print("DF V6 Phase 3B — EDL+ORCU λ Sweep (log-barrier)"); print("="*70)
LAM_VALS=[0.05,0.10,0.20]
p3b=[]
for lam in LAM_VALS:
    for seed in SWEEP_SEEDS:
        print(f"
  EDL+ORCU λ={lam} (log-barrier) s{seed}")
        out=os.path.join(OUTPUT_DIR,f"v6_p3b_orcu_lam{lam}_s{seed}")
        m,_=train_model_v6("df",DATA_ROOT,out,mode="edl_orcu",lambda_reg=lam,use_log_barrier=True,epochs=EPOCHS,seed=seed)
        r={"lambda_reg":lam,"seed":seed,"regularizer":"log_barrier","acc":m["acc"],"macro_f1":m["macro_f1"],"qwk":m["qwk"],"ece":m["ece"],"pct_unimodal":m["pct_unimodal"]}
        if "per_class_f1" in m: r["per_class_f1"]=m["per_class_f1"]
        p3b.append(r)
with open(os.path.join(OUTPUT_DIR,"v6_phase3b_orcu_sweep.json"),"w") as f: json.dump(p3b,f,indent=2)
for lam in LAM_VALS:
    s=[r for r in p3b if r["lambda_reg"]==lam]
    print(f"λ={lam:.2f}: Acc={np.mean([x["acc"] for x in s]):.4f}  %Unim={np.mean([x["pct_unimodal"] for x in s]):.2%}")
print("Phase 3B done.")

## 10. Phase 4: 5-Seed Stability (25 runs, ~7.5h)

In [ ]:
print("="*70); print("DF V6 Phase 4 — 5-Seed Stability"); print("="*70)
p4={}
for mode in MODES_V6:
    p4[mode]={}
    for seed in SEEDS_V6:
        print(f"
  {mode} | s{seed}")
        out=os.path.join(OUTPUT_DIR,f"v6_p4_{mode}_s{seed}")
        m,_=train_model_v6("df",DATA_ROOT,out,mode=mode,epochs=EPOCHS,seed=seed)
        p4[mode][str(seed)]={k:v for k,v in m.items() if isinstance(v,(float,int,bool,list))}
with open(os.path.join(OUTPUT_DIR,"v6_phase4_multiseed.json"),"w") as f: json.dump(p4,f,indent=2)
print("
--- Phase 4 Results ---")
for mode in MODES_V6:
    accs=[p4[mode][str(s)]["acc"] for s in SEEDS_V6]
    print(f"{mode:<14} Acc={np.mean(accs):.4f}+/-{np.std(accs):.4f}")
print("Phase 4 done.")

## 11. Phase 5: 10-Fold CV — PAPER RESULT (50 folds, ~15h)

In [ ]:
print("#"*60); print("# DF V6 Phase 5: 10-Fold CV"); print("#"*60)

from torch.utils.data import DataLoader, Subset
from sklearn.model_selection import StratifiedKFold
from scipy import stats

def run_cv(data_root, cfg, n_folds=10, epochs=100, seed=42):
    ds=DFDataset(data_root,split="train",split_seed=seed)
    labels=np.array([ds[i][1] for i in range(len(ds))])
    skf=StratifiedKFold(n_splits=n_folds,shuffle=True,random_state=seed)
    mnames=[m[0] for m in cfg]; fres={mn:[] for mn in mnames}
    for fi,(tr,vl) in enumerate(skf.split(np.zeros(len(ds)),labels)):
        print(f"
Fold {fi+1}/{n_folds}")
        tr_full=DFDataset(data_root,split="train",split_seed=seed)
        vl_full=DFDataset(data_root,split="train",split_seed=seed)
        tr_full.transform=get_transforms("df",is_train=True)
        vl_full.transform=get_transforms("df",is_train=False)
        tr_sub=Subset(tr_full,tr); vl_sub=Subset(vl_full,vl)
        tds=DFDataset(data_root,split="test"); tds.transform=get_transforms("df",is_train=False)
        tr_ldr=DataLoader(tr_sub,batch_size=32,shuffle=True,num_workers=2,pin_memory=True)
        vl_ldr=DataLoader(vl_sub,batch_size=32,shuffle=False,num_workers=2,pin_memory=True)
        tst_ldr=DataLoader(tds,batch_size=32,shuffle=False,num_workers=2,pin_memory=True)
        for mn,kw in cfg:
            print(f"  [{mn}]")
            set_seed(seed+fi*100)
            model=build_model(task="df",mode=mn); model.to(device)
            kl=kw.get("kl_lambda",V6_KL); kc=kw.get("kl_anneal_cap",V6_CAP)
            if mn=="ce":
                def cr(a,z,t,e): loss=F.nll_loss(F.log_softmax(z,dim=-1),t); return loss,{"stage":0,"L_ce":loss.item()}
            elif mn=="cumulative":
                from losses.cumulative_loss import CumulativeLoss
                cf=CumulativeLoss(num_classes=4)
                def cr(a,z,t,e):
                    ol=model.ordinal_logits if hasattr(model,"ordinal_logits") else z
                    loss=cf(ol,t); return loss,{"stage":0,"L_cum":loss.item()}
            elif mn=="sord":
                from losses.orcu_loss import ORCULoss
                sf=ORCULoss(num_classes=4,t=3.0,lambda_reg=0.0)
                def cr(a,z,t,e): loss=sf(z,t); return loss,{"stage":0,"L_sord":loss.item()}
            elif mn=="edl":
                from losses.edl_loss import EDLLoss
                ef=EDLLoss(num_classes=4,kl_lambda=kl,kl_anneal_cap=kc)
                def cr(a,z,t,e): loss=ef(a,t,e,epochs); return loss,{"stage":0,"L_edl":loss.item()}
            else:
                cr=CombinedLoss(num_classes=4,lambda_orcu=kw.get("lambda_orcu",0.10),lambda_kl=kl,kl_anneal_cap=kc,orcu_t=3.0,
                                orcu_lambda_reg=kw.get("lambda_reg",0.10),stage_1_epochs=3,stage_2_epochs=10,total_epochs=epochs)
            bb,hd=[],[]
            for n,p in model.named_parameters():
                if not p.requires_grad: continue
                (bb if n.startswith("backbone") else hd).append(p)
            opt=optim.AdamW([{"params":bb,"lr":1e-4},{"params":hd,"lr":1e-3}],weight_decay=0.05)
            ws=LinearLR(opt,start_factor=0.1,total_iters=5)
            cs=CosineAnnealingLR(opt,T_max=epochs-5)
            sch=SequentialLR(opt,schedulers=[ws,cs],milestones=[5])
            es=EarlyStopping(patience=kw.get("patience",15),mode="max")
            bva,bst=0.0,None
            @torch.no_grad()
            def ev(m,ldr):
                m.eval(); aa,zz,tt=[],[],[]
                for im,tg in ldr:
                    im,tg=im.to(device),tg.to(device)
                    a,z_=m(im)
                    if a is not None: aa.append(a.cpu())
                    zz.append(z_.cpu()); tt.append(tg.cpu())
                a=torch.cat(aa) if aa else None
                return compute_metrics(a,torch.cat(zz),torch.cat(tt))
            for ep in range(epochs):
                model.train()
                for im,tg in tr_ldr:
                    im,tg=im.to(device),tg.to(device)
                    alpha,z=model(im); loss,_=cr(alpha,z,tg,ep)
                    opt.zero_grad(); loss.backward()
                    torch.nn.utils.clip_grad_norm_(model.parameters(),max_norm=1.0)
                    opt.step()
                sch.step()
                vm=ev(model,vl_ldr)
                if vm["acc"]>bva: bva=vm["acc"]; bst=copy.deepcopy(model.state_dict())
                if es(vm["acc"],ep,model): break
            es.restore(model); tm=ev(model,tst_ldr)
            fres[mn].append({"fold":fi,**{k:v for k,v in tm.items() if isinstance(v,(float,int,bool,list,np.ndarray))}})
            print(f"    Acc={tm["acc"]:.4f} F1={tm["macro_f1"]:.4f} QWK={tm["qwk"]:.4f}")
    summary={}
    for mn in mnames:
        agg={}
        for mk in ["acc","macro_f1","weighted_f1","qwk","ece","pct_unimodal"]:
            vals=[f[mk] for f in fres[mn] if mk in f]
            if vals: agg[mk]={"mean":float(np.mean(vals)),"std":float(np.std(vals))}
        if "per_class_f1" in fres[mn][0]:
            pf=np.array([f["per_class_f1"] for f in fres[mn]])
            agg["per_class_f1"]={"mean":pf.mean(0).tolist(),"std":pf.std(0).tolist()}
        summary[mn]=agg
    sig={}
    for i,m1 in enumerate(mnames):
        for j,m2 in enumerate(mnames):
            if i>=j: continue
            pk=f"{m1}_vs_{m2}"; sig[pk]={}
            for mk in ["acc","macro_f1","weighted_f1","qwk","ece"]:
                v1=[f[mk] for f in fres[m1]]; v2=[f[mk] for f in fres[m2]]
                t,p=stats.ttest_rel(v1,v2)
                sig[pk][mk]={"t_statistic":float(t),"p_value":float(p),"significant":bool(p<0.05)}
    return {"task":"df","k":n_folds,"methods":mnames,"summary":summary,"significance":sig}

cv_cfg=[("ce",{"patience":15}),("cumulative",{"patience":15}),("sord",{"patience":15}),
        ("edl",{"kl_lambda":0.07,"kl_anneal_cap":0.5,"patience":25}),
        ("edl_orcu",{"kl_lambda":0.07,"lambda_orcu":0.10,"lambda_reg":0.10,"kl_anneal_cap":0.5,"patience":15})]
cv_res=run_cv(DATA_ROOT,cfg=cv_cfg,n_folds=10,epochs=EPOCHS)

def cvt(o):
    if isinstance(o,dict): return {k:cvt(v) for k,v in o.items()}
    elif isinstance(o,list): return [cvt(v) for v in o]
    elif isinstance(o,(np.floating,np.integer)): return float(o)
    elif isinstance(o,np.ndarray): return o.tolist()
    return o
with open(os.path.join(OUTPUT_DIR,"v6_phase5_cv.json"),"w") as f: json.dump(cvt(cv_res),f,indent=2)

print("
"+"="*90); print("CV RESULTS — Paper Core"); print("="*90)
for mn,agg in cv_res["summary"].items():
    print(f"{mn:<14}: Acc={agg["acc"]["mean"]:.4f}+/-{agg["acc"]["std"]:.4f}  QWK={agg["qwk"]["mean"]:.4f}+/-{agg["qwk"]["std"]:.4f}  ECE={agg["ece"]["mean"]:.4f}")
edl_agg=cv_res["summary"].get("edl",{})
if edl_agg:
    a=edl_agg["acc"]["mean"]; q=edl_agg["qwk"]["mean"]
    tier="S—Paper SOTA!" if a>=0.82 else ("A—Beats competitors!" if a>=0.80 else "B—Competitive")
    print(f"
EDL CV: Acc={a:.4f} QWK={q:.4f} → Tier: {tier}")
print("Phase 5 done.")

## 12. Phase 6: A6 Ablation — Log-Barrier vs Hinge (6 runs, ~2h)

In [ ]:
print("="*70); print("DF V6 Phase 6 — A6: Log-Barrier vs Hinge"); print("="*70)
p6=[]
for rn,ulb in [("log_barrier",True),("hinge",False)]:
    for seed in SWEEP_SEEDS:
        print(f"
  EDL+ORCU {rn} s{seed}")
        out=os.path.join(OUTPUT_DIR,f"v6_p6_{rn}_s{seed}")
        m,_=train_model_v6("df",DATA_ROOT,out,mode="edl_orcu",use_log_barrier=ulb,epochs=EPOCHS,seed=seed)
        r={"regularizer":rn,"seed":seed,"acc":m["acc"],"macro_f1":m["macro_f1"],"qwk":m["qwk"],"ece":m["ece"],"pct_unimodal":m["pct_unimodal"]}
        if "per_class_f1" in m: r["per_class_f1"]=m["per_class_f1"]
        p6.append(r)
with open(os.path.join(OUTPUT_DIR,"v6_phase6_ablation.json"),"w") as f: json.dump(p6,f,indent=2)
print("
--- A6 Results ---")
for reg in ["log_barrier","hinge"]:
    s=[r for r in p6 if r["regularizer"]==reg]
    if s: print(f"{reg:<16}: Acc={np.mean([x["acc"] for x in s]):.4f}  %Unim={np.mean([x["pct_unimodal"] for x in s]):.2%}  ECE={np.mean([x["ece"] for x in s]):.4f}")
lb_u=np.mean([r["pct_unimodal"] for r in p6 if r["regularizer"]=="log_barrier"])
hg_u=np.mean([r["pct_unimodal"] for r in p6 if r["regularizer"]=="hinge"])
print(f"
Δ %Unimodal: {lb_u-hg_u:+.2%}  → {"✅ Log-barrier wins!" if lb_u>hg_u else "⚠ Check"}")
print("Phase 6 done.")

## 13. Final Summary & Export

In [ ]:
print("
"+"="*90); print("V6 vs COMPETITORS"); print("="*90)
print(f"
{"Method":<24} {"Acc":>10} {"Params":>10} {"Uncertainty":>14} {"Ordinal":>12}")
print("-"*78)
print(f"{"MLTrMR (2025)":<24} {"80.19%":>10} {"556M":>10} {"None":>14} {"None":>12}")
print(f"{"LD2Net (2026)":<24} {"80.00%":>10} {"3.3M":>10} {"None":>14} {"None":>12}")

all_s={}
for fn,k in [("v6_phase2_baseline.json","p2"),("v6_phase3a_kl_sweep.json","p3a"),("v6_phase3b_orcu_sweep.json","p3b"),
             ("v6_phase4_multiseed.json","p4"),("v6_phase5_cv.json","p5"),("v6_phase6_ablation.json","p6")]:
    fp=os.path.join(OUTPUT_DIR,fn)
    if os.path.exists(fp):
        with open(fp) as f: all_s[k]=json.load(f)

if "p5" in all_s:
    for mn,agg in all_s["p5"]["summary"].items():
        unc="Dirichlet" if mn in ("edl","edl_orcu") else "—"
        ord_t="Log-barrier" if mn=="edl_orcu" else ("Implicit" if mn=="edl" else "—")
        print(f"{"Ours "+mn.upper()+" (V6)":<24} {agg["acc"]["mean"]:.2%}+/-{agg["acc"]["std"]:.2%}  {"86M":>10} {unc:>14} {ord_t:>12}")

# Master summary
def cvt(o):
    if isinstance(o,dict): return {k:cvt(v) for k,v in o.items()}
    elif isinstance(o,list): return [cvt(v) for v in o]
    elif isinstance(o,(np.floating,np.integer)): return float(o)
    elif isinstance(o,np.ndarray): return o.tolist()
    return o
if all_s:
    with open(os.path.join(OUTPUT_DIR,"v6_master_summary.json"),"w") as f: json.dump(cvt(all_s),f,indent=2)

# Export
import shutil as _sh
DST="/kaggle/working/df_exp_result"
if os.path.exists(DST): _sh.rmtree(DST)
os.makedirs(DST,exist_ok=True)
exported=0
for fn in ["v6_phase2_baseline.json","v6_phase3a_kl_sweep.json","v6_phase3b_orcu_sweep.json",
           "v6_phase4_multiseed.json","v6_phase5_cv.json","v6_phase6_ablation.json","v6_master_summary.json"]:
    src=os.path.join(OUTPUT_DIR,fn)
    if os.path.exists(src): _sh.copy2(src,os.path.join(DST,fn)); exported+=1
for f in sorted(os.listdir(OUTPUT_DIR)):
    if f.endswith(".npz") and f.startswith("v6_"): _sh.copy2(os.path.join(OUTPUT_DIR,f),os.path.join(DST,f)); exported+=1
if exported>0:
    md={"title":"Fluorosis DF V6 Results","id":"hgxiao/fluorosis-df-v6-results","licenses":[{"name":"CC0-1.0"}]}
    with open(os.path.join(DST,"dataset-metadata.json"),"w") as f: json.dump(md,f)
    from pathlib import Path
    sz=sum(os.path.getsize(os.path.join(r,f)) for r,_,fs in os.walk(DST) for f in fs)
    print(f"
Export: {sum(1 for _ in Path(DST).rglob("*") if _.is_file())} files, {sz/(1024*1024):.1f}MB")
    print("Upload: kaggle datasets create -p /kaggle/working/df_exp_result --dir-mode tar")
print("
=== V6 EXPERIMENTS COMPLETE ===")